# Simulador de Verificação de Decolagem - Sistema Aeroespacial

## Sistema Automatizado de Verificação de Segurança

Este notebook implementa um sistema completo de verificação de decolagem que:
- Coleta dados de múltiplos sensores
- Valida integridade estrutural e energética
- Calcula autonomia energética
- Toma decisão de decolagem ou aborto

---

## Célula 1: Importações de Bibliotecas

In [1]:
import pandas as pd
import numpy as np
import random
from datetime import datetime, timedelta
import warnings

# Suprimir avisos desnecessários
warnings.filterwarnings('ignore')

## Célula 2: Configuração Inicial e Parâmetros do Sistema

In [2]:
# ═══════════════════════════════════════════════════════════════════
# CONFIGURAÇÃO DO SISTEMA DE VERIFICAÇÃO DE DECOLAGEM
# ═══════════════════════════════════════════════════════════════════

# LIMITES DE TEMPERATURA (em Celsius)
TEMP_INTERNA_MIN = -10
TEMP_INTERNA_MAX = 50
TEMP_EXTERNA_MIN = -50
TEMP_EXTERNA_MAX = 30

# LIMITES DE ENERGIA
ENERGIA_MINIMA = 80  # Mínimo 80% para decolagem segura
ENERGIA_MAXIMA = 100

# LIMITES DE PRESSÃO (PSI - Pounds per Square Inch)
PRESSAO_MINIMA = 400
PRESSAO_MAXIMA = 600

# AUTONOMIA MÍNIMA REQUERIDA (em minutos)
AUTONOMIA_MINIMA = 60

# CAPACIDADE ENERGÉTICA
CAPACIDADE_BATERIA = 100  # kWh
CONSUMO_DECOLAGEM = 5000  # W (5 kW)
PERDAS_ENERGETICAS = 0.15  # 15% de perdas

# SIMULAÇÃO
NUM_CICLOS = 30  # Número de leituras de sensores
MODULOS_CRITICOS = ['Propulsão', 'Aviônica', 'Combustível', 'Hidráulico', 'Eletrônico', 'Sensores']

print("\n" + "="*70)
print("PARÂMETROS DO SISTEMA CONFIGURADOS")
print("="*70)
print(f"\nTemperatura Interna: {TEMP_INTERNA_MIN}°C a {TEMP_INTERNA_MAX}°C")
print(f"Temperatura Externa: {TEMP_EXTERNA_MIN}°C a {TEMP_EXTERNA_MAX}°C")
print(f"Energia Mínima: {ENERGIA_MINIMA}%")
print(f"Pressão Mínima: {PRESSAO_MINIMA} PSI")
print(f"Autonomia Mínima: {AUTONOMIA_MINIMA} minutos")
print(f"Capacidade de Bateria: {CAPACIDADE_BATERIA} kWh")
print(f"Número de Ciclos: {NUM_CICLOS}")
print(f"Módulos Críticos: {len(MODULOS_CRITICOS)} monitorados")
print("="*70 + "\n")


PARÂMETROS DO SISTEMA CONFIGURADOS

Temperatura Interna: -10°C a 50°C
Temperatura Externa: -50°C a 30°C
Energia Mínima: 80%
Pressão Mínima: 400 PSI
Autonomia Mínima: 60 minutos
Capacidade de Bateria: 100 kWh
Número de Ciclos: 30
Módulos Críticos: 6 monitorados



## Célula 3: Geração de Dados Simulados dos Sensores

In [3]:
def gerar_dados_sensores():
    """
    Gera dados simulados de sensores com distribuição realista.
    Simula leitura real de uma aeronave.
    """
    dados = []
    
    for ciclo in range(NUM_CICLOS):
        # Temperatura Interna: valores normalmente distribuídos em torno de 25°C
        temp_interna = np.random.normal(25, 11)  # Média 25°C, desvio 11°C
        
        # Temperatura Externa: varia bastante
        temp_externa = np.random.normal(-5, 15)  # Média -5°C, desvio 15°C
        
        # Integridade Estrutural: 98% de chance de estar íntegra
        integridade = 1 if random.random() > 0.02 else 0
        
        # Nível de Energia: começa alto e degrada ligeiramente
        energia = np.random.normal(95 - ciclo * 0.3, 4)  # Degrada com ciclos
        energia = max(0, min(ENERGIA_MAXIMA, energia))
        
        # Pressão dos Tanques: varia em torno de 500 PSI
        pressao = np.random.normal(500, 43)  # Média 500, desvio 43
        pressao = max(PRESSAO_MINIMA - 100, min(PRESSAO_MAXIMA + 100, pressao))
        
        # Status dos Módulos Críticos: 1 = Operacional, 0 = Falha
        modulos = {modulo: 1 if random.random() > 0.005 else 0 
                   for modulo in MODULOS_CRITICOS}  # 99,5% de sucesso por módulo
        
        # Timestamp
        timestamp = datetime.now() + timedelta(seconds=ciclo)
        
        # Adicionar linha aos dados
        linha = {
            'Ciclo': ciclo + 1,
            'Timestamp': timestamp,
            'Temperatura_Interna_°C': round(temp_interna, 2),
            'Temperatura_Externa_°C': round(temp_externa, 2),
            'Integridade_Estrutural': integridade,
            'Energia_%': round(energia, 2),
            'Pressao_Tanques_PSI': round(pressao, 2),
        }
        
        # Adicionar status de cada módulo
        for modulo, status in modulos.items():
            linha[f'Modulo_{modulo}'] = status
        
        dados.append(linha)
    
    # Criar DataFrame
    df = pd.DataFrame(dados)
    return df

# Gerar dados
print("\n" + "="*70)
print("GERANDO DADOS SIMULADOS DOS SENSORES")
print("="*70)
print(f"Gerando {NUM_CICLOS} ciclos de leitura...\n")

df_sensores = gerar_dados_sensores()

print(f"{len(df_sensores)} ciclos de leitura gerados com sucesso!\n")
print("Primeiras 5 leituras:")
print(df_sensores.head())


GERANDO DADOS SIMULADOS DOS SENSORES
Gerando 30 ciclos de leitura...

30 ciclos de leitura gerados com sucesso!

Primeiras 5 leituras:
   Ciclo                  Timestamp  Temperatura_Interna_°C  \
0      1 2026-09-16 01:12:54.317193                   25.20   
1      2 2026-09-16 01:12:55.317215                   23.01   
2      3 2026-09-16 01:12:56.317222                   25.58   
3      4 2026-09-16 01:12:57.317228                   33.25   
4      5 2026-09-16 01:12:58.317233                   18.62   

   Temperatura_Externa_°C  Integridade_Estrutural  Energia_%  \
0                   15.38                       1      89.84   
1                  -17.38                       1      89.73   
2                   -7.80                       1      97.07   
3                   -3.00                       1      91.01   
4                   -8.32                       1      91.62   

   Pressao_Tanques_PSI  Modulo_Propulsão  Modulo_Aviônica  Modulo_Combustível  \
0               526

## Célula 4: Interpretação e Análise Inicial dos Dados

In [4]:
print("\n" + "="*70)
print("INTERPRETAÇÃO DOS DADOS COLETADOS")
print("="*70)

# Usar a última leitura (mais atual)
ultima_leitura = df_sensores.iloc[-1]

print(f"\n LEITURA FINAL DO SISTEMA (Ciclo {ultima_leitura['Ciclo']})")
print("-" * 70)

# Temperatura
print(f"\n TEMPERATURA:")
print(f"Interna: {ultima_leitura['Temperatura_Interna_°C']}°C ", end="")
if TEMP_INTERNA_MIN <= ultima_leitura['Temperatura_Interna_°C'] <= TEMP_INTERNA_MAX:
    print("DENTRO DO LIMITE")
else:
    print("FORA DO LIMITE")
    
print(f"Externa: {ultima_leitura['Temperatura_Externa_°C']}°C ", end="")
if TEMP_EXTERNA_MIN <= ultima_leitura['Temperatura_Externa_°C'] <= TEMP_EXTERNA_MAX:
    print("DENTRO DO LIMITE")
else:
    print("FORA DO LIMITE")

# Integridade
print(f"\n INTEGRIDADE ESTRUTURAL:")
if ultima_leitura['Integridade_Estrutural'] == 1:
    print(f"Status: ÍNTEGRA")
else:
    print(f"Status: COMPROMETIDA")

# Energia
print(f"\n NÍVEL DE ENERGIA:")
print(f"Carga Atual: {ultima_leitura['Energia_%']}%")
if ultima_leitura['Energia_%'] >= ENERGIA_MINIMA:
    print(f"Status: SUFICIENTE (Mínimo: {ENERGIA_MINIMA}%)")
else:
    print(f"Status: BAIXA (Mínimo: {ENERGIA_MINIMA}%)")

# Pressão
print(f"\n PRESSÃO DOS TANQUES:")
print(f"Pressão: {ultima_leitura['Pressao_Tanques_PSI']} PSI")
if PRESSAO_MINIMA <= ultima_leitura['Pressao_Tanques_PSI'] <= PRESSAO_MAXIMA:
    print(f"Status: ADEQUADA (Faixa: {PRESSAO_MINIMA} a {PRESSAO_MAXIMA} PSI)")
else:
    print(f"Status: FORA DA FAIXA (Faixa: {PRESSAO_MINIMA} a {PRESSAO_MAXIMA} PSI)")

# Módulos Críticos
print(f"\n MÓDULOS CRÍTICOS:")
modulos_status = []
for modulo in MODULOS_CRITICOS:
    col_name = f'Modulo_{modulo}'
    status = "OPERACIONAL" if ultima_leitura[col_name] == 1 else "FALHA"
    print(f"{modulo}: {status}")
    modulos_status.append(ultima_leitura[col_name])

modulos_ok = sum(modulos_status)
print(f"\n Resumo: {modulos_ok}/{len(MODULOS_CRITICOS)} módulos operacionais")

# Estatísticas gerais
print(f"\n" + "="*70)
print(f"ESTATÍSTICAS DOS {NUM_CICLOS} CICLOS")
print("="*70)
print(f"\n Temperatura Interna:")
print(f"Mínima: {df_sensores['Temperatura_Interna_°C'].min():.2f}°C")
print(f"Máxima: {df_sensores['Temperatura_Interna_°C'].max():.2f}°C")
print(f"Média: {df_sensores['Temperatura_Interna_°C'].mean():.2f}°C")

print(f"\n Energy Levels:")
print(f"Inicial: {df_sensores['Energia_%'].iloc[0]:.2f}%")
print(f"Final: {df_sensores['Energia_%'].iloc[-1]:.2f}%")
print(f"Mínima: {df_sensores['Energia_%'].min():.2f}%")
print(f"Consumo Total: {df_sensores['Energia_%'].iloc[0] - df_sensores['Energia_%'].iloc[-1]:.2f}%")

print(f"\n Pressão dos Tanques:")
print(f"Mínima: {df_sensores['Pressao_Tanques_PSI'].min():.2f} PSI")
print(f"Máxima: {df_sensores['Pressao_Tanques_PSI'].max():.2f} PSI")
print(f"Média: {df_sensores['Pressao_Tanques_PSI'].mean():.2f} PSI")

print("\n" + "="*70 + "\n")


INTERPRETAÇÃO DOS DADOS COLETADOS

 LEITURA FINAL DO SISTEMA (Ciclo 30)
----------------------------------------------------------------------

 TEMPERATURA:
Interna: 15.91°C DENTRO DO LIMITE
Externa: -21.0°C DENTRO DO LIMITE

 INTEGRIDADE ESTRUTURAL:
Status: ÍNTEGRA

 NÍVEL DE ENERGIA:
Carga Atual: 86.87%
Status: SUFICIENTE (Mínimo: 80%)

 PRESSÃO DOS TANQUES:
Pressão: 541.75 PSI
Status: ADEQUADA (Faixa: 400 a 600 PSI)

 MÓDULOS CRÍTICOS:
Propulsão: OPERACIONAL
Aviônica: OPERACIONAL
Combustível: OPERACIONAL
Hidráulico: OPERACIONAL
Eletrônico: OPERACIONAL
Sensores: OPERACIONAL

 Resumo: 6/6 módulos operacionais

ESTATÍSTICAS DOS 30 CICLOS

 Temperatura Interna:
Mínima: 0.45°C
Máxima: 52.88°C
Média: 29.35°C

 Energy Levels:
Inicial: 89.84%
Final: 86.87%
Mínima: 86.04%
Consumo Total: 2.97%

 Pressão dos Tanques:
Mínima: 443.92 PSI
Máxima: 555.37 PSI
Média: 505.46 PSI




## Célula 5: Algoritmo de Verificação de Decolagem

In [5]:
def verificar_decolagem(df, ultima_leitura):
    """
    Algoritmo principal de verificação de decolagem.
    Implementa múltiplas camadas de validação.
    
    Retorna: (pode_decolar: bool, motivos_aborto: list, autonomia: float)
    """
    
    motivos_aborto = []
    print("\n" + "="*70)
    print("ALGORITMO DE VERIFICAÇÃO DE DECOLAGEM - INICIADO")
    print("="*70 + "\n")
    
    # Passo 1: Verificar Temperatura Interna
    print("\n[1/6] Verificando Temperatura Interna...")
    temp_interna = ultima_leitura['Temperatura_Interna_°C']
    if TEMP_INTERNA_MIN <= temp_interna <= TEMP_INTERNA_MAX:
        print(f"OK - {temp_interna}°C (Limite: {TEMP_INTERNA_MIN}°C a {TEMP_INTERNA_MAX}°C)")
    else:
        print(f"FALHA - {temp_interna}°C (Limite: {TEMP_INTERNA_MIN}°C a {TEMP_INTERNA_MAX}°C)")
        motivos_aborto.append(f"Temperatura interna fora do limite: {temp_interna}°C")
    
    # Passo 2: Verificar Temperatura Externa
    print("\n[2/6] Verificando Temperatura Externa...")
    temp_externa = ultima_leitura['Temperatura_Externa_°C']
    if TEMP_EXTERNA_MIN <= temp_externa <= TEMP_EXTERNA_MAX:
        print(f"OK - {temp_externa}°C (Limite: {TEMP_EXTERNA_MIN}°C a {TEMP_EXTERNA_MAX}°C)")
    else:
        print(f"FALHA - {temp_externa}°C (Limite: {TEMP_EXTERNA_MIN}°C a {TEMP_EXTERNA_MAX}°C)")
        motivos_aborto.append(f"Temperatura externa fora do limite: {temp_externa}°C")
    
    # Passo 3: Verificar Integridade Estrutural
    print("\n[3/6] Verificando Integridade Estrutural...")
    integridade = ultima_leitura['Integridade_Estrutural']
    if integridade == 1:
        print(f"OK - Estrutura íntegra")
    else:
        print(f"FALHA - Estrutura comprometida!")
        motivos_aborto.append("Integridade estrutural comprometida")
    
    # Passo 4: Verificar Nível de Energia
    print("\n[4/6] Verificando Nível de Energia...")
    energia = ultima_leitura['Energia_%']
    if energia >= ENERGIA_MINIMA:
        print(f"OK - {energia}% (Mínimo: {ENERGIA_MINIMA}%)")
    else:
        print(f"FALHA - {energia}% (Mínimo: {ENERGIA_MINIMA}%)")
        motivos_aborto.append(f"Energia insuficiente: {energia}% (Mínimo: {ENERGIA_MINIMA}%)")
    
    # Passo 5: Verificar Pressão dos Tanques
    print("\n[5/6] Verificando Pressão dos Tanques...")
    pressao = ultima_leitura['Pressao_Tanques_PSI']
    if PRESSAO_MINIMA <= pressao <= PRESSAO_MAXIMA:
        print(f"OK - {pressao} PSI (Faixa: {PRESSAO_MINIMA} a {PRESSAO_MAXIMA} PSI)")
    else:
        print(f"FALHA - {pressao} PSI (Faixa: {PRESSAO_MINIMA} a {PRESSAO_MAXIMA} PSI)")
        motivos_aborto.append(f"Pressão fora da faixa: {pressao} PSI (Faixa: {PRESSAO_MINIMA} a {PRESSAO_MAXIMA} PSI)")
    
    # Passo 6: Verificar Módulos Críticos
    print("\n[6/6] Verificando Módulos Críticos...")
    modulos_falhos = []
    for modulo in MODULOS_CRITICOS:
        col_name = f'Modulo_{modulo}'
        status = ultima_leitura[col_name]
        if status == 0:
            modulos_falhos.append(modulo)
    
    if len(modulos_falhos) == 0:
        print(f"OK - Todos os {len(MODULOS_CRITICOS)} módulos operacionais")
    else:
        print(f"FALHA - Módulos inoperacionais: {', '.join(modulos_falhos)}")
        motivos_aborto.append(f"Módulos críticos inoperacionais: {', '.join(modulos_falhos)}")
    
    # Cálculo de Autonomia Energética (será feito na próxima célula)
    # Por enquanto, estimamos
    energia_kwh = (energia / 100) * CAPACIDADE_BATERIA
    consumo_kw = CONSUMO_DECOLAGEM / 1000  # Converter para kW
    perdas = consumo_kw * PERDAS_ENERGETICAS
    consumo_total_kw = consumo_kw + perdas
    autonomia_horas = energia_kwh / consumo_total_kw if consumo_total_kw > 0 else 0
    autonomia_minutos = autonomia_horas * 60
    
    # Passo 7: Verificar Autonomia Energética
    print("\n[EXTRA] Verificando Autonomia Energética...")
    if autonomia_minutos >= AUTONOMIA_MINIMA:
        print(f"OK - {autonomia_minutos:.2f} minutos (Mínimo: {AUTONOMIA_MINIMA} minutos)")
    else:
        print(f"FALHA - {autonomia_minutos:.2f} minutos (Mínimo: {AUTONOMIA_MINIMA} minutos)")
        motivos_aborto.append(f"Autonomia insuficiente: {autonomia_minutos:.2f} min (Mínimo: {AUTONOMIA_MINIMA} min)")
    
    # Decisão final
    pode_decolar = len(motivos_aborto) == 0
    
    return pode_decolar, motivos_aborto, autonomia_minutos

# Executar verificação
pode_decolar, motivos, autonomia = verificar_decolagem(df_sensores, df_sensores.iloc[-1])

print("\n" + "="*70)
print("RESUMO DA VERIFICAÇÃO")
print("="*70)

if pode_decolar:
    print("\n TODOS OS SISTEMAS OK")
    print(f"Autonomia Estimada: {autonomia:.2f} minutos")
else:
    print(f"\n FALHAS DETECTADAS ({len(motivos)} motivo(s)):")
    for i, motivo in enumerate(motivos, 1):
        print(f"{i}. {motivo}")

print("\n" + "="*70 + "\n")


ALGORITMO DE VERIFICAÇÃO DE DECOLAGEM - INICIADO


[1/6] Verificando Temperatura Interna...
OK - 15.91°C (Limite: -10°C a 50°C)

[2/6] Verificando Temperatura Externa...
OK - -21.0°C (Limite: -50°C a 30°C)

[3/6] Verificando Integridade Estrutural...
OK - Estrutura íntegra

[4/6] Verificando Nível de Energia...
OK - 86.87% (Mínimo: 80%)

[5/6] Verificando Pressão dos Tanques...
OK - 541.75 PSI (Faixa: 400 a 600 PSI)

[6/6] Verificando Módulos Críticos...
OK - Todos os 6 módulos operacionais

[EXTRA] Verificando Autonomia Energética...
OK - 906.47 minutos (Mínimo: 60 minutos)

RESUMO DA VERIFICAÇÃO

 TODOS OS SISTEMAS OK
Autonomia Estimada: 906.47 minutos




## Célula 5b: Cenários de Teste (todos os ciclos)

In [6]:
import io, contextlib

resultados = []
for _, linha in df_sensores.iterrows():
    with contextlib.redirect_stdout(io.StringIO()):   # silencia o log detalhado de cada ciclo
        ok, mot, _ = verificar_decolagem(df_sensores, linha)
    resultados.append({'Ciclo': int(linha['Ciclo']),
                       'Decisão': 'PRONTO' if ok else 'ABORTADA',
                       'Motivo': '; '.join(mot) if mot else '-'})

df_cenarios = pd.DataFrame(resultados)

print("\n" + "="*70)
print(f"CENÁRIOS DE TESTE - {len(df_cenarios)} CICLOS AVALIADOS")
print("="*70 + "\n")
print(df_cenarios.to_string(index=False))
print(f"\nAprovados: {(df_cenarios['Decisão']=='PRONTO').sum()} | "
      f"Abortados: {(df_cenarios['Decisão']=='ABORTADA').sum()}")
print("\n" + "="*70 + "\n")



CENÁRIOS DE TESTE - 30 CICLOS AVALIADOS

 Ciclo  Decisão                                      Motivo
     1   PRONTO                                           -
     2   PRONTO                                           -
     3   PRONTO                                           -
     4   PRONTO                                           -
     5   PRONTO                                           -
     6   PRONTO                                           -
     7 ABORTADA Temperatura interna fora do limite: 52.88°C
     8   PRONTO                                           -
     9   PRONTO                                           -
    10   PRONTO                                           -
    11   PRONTO                                           -
    12   PRONTO                                           -
    13   PRONTO                                           -
    14   PRONTO                                           -
    15   PRONTO                                           

## Célula 6: Análise Energética Detalhada

In [7]:
def analisar_energia(df, ultima_leitura):
    """
    Realiza análise energética completa da aeronave.
    """
    
    print("\n" + "="*70)
    print("ANÁLISE ENERGÉTICA DETALHADA")
    print("="*70)
    
    # Dados base
    energia_percentual = ultima_leitura['Energia_%']
    energia_kwh = (energia_percentual / 100) * CAPACIDADE_BATERIA
    
    print(f"\n CAPACIDADE DO SISTEMA:")
    print(f"Capacidade Total: {CAPACIDADE_BATERIA} kWh")
    print(f"Carga Atual: {energia_kwh:.2f} kWh ({energia_percentual:.2f}%)")
    
    # Consumo na decolagem
    consumo_decolagem_kw = CONSUMO_DECOLAGEM / 1000  # Converter W para kW
    
    print(f"\n CONSUMO NA DECOLAGEM:")
    print(f"Potência Nominal: {CONSUMO_DECOLAGEM} W ({consumo_decolagem_kw} kW)")
    
    # Perdas energéticas
    perdas_kw = consumo_decolagem_kw * PERDAS_ENERGETICAS
    print(f"\n PERDAS ENERGÉTICAS:")
    print(f"Taxa de Perda: {PERDAS_ENERGETICAS * 100:.1f}%")
    print(f"Perda Absoluta: {perdas_kw:.3f} kW")
    
    # Consumo total
    consumo_total_kw = consumo_decolagem_kw + perdas_kw
    print(f"\n CONSUMO TOTAL:")
    print(f"Consumo + Perdas: {consumo_total_kw:.3f} kW")
    
    # Autonomia
    if consumo_total_kw > 0:
        autonomia_horas = energia_kwh / consumo_total_kw
        autonomia_minutos = autonomia_horas * 60
        autonomia_segundos = autonomia_horas * 3600
    else:
        autonomia_horas = autonomia_minutos = autonomia_segundos = 0
    
    print(f"\n AUTONOMIA ESTIMADA:")
    print(f"{autonomia_horas:.2f} horas")
    print(f"{autonomia_minutos:.2f} minutos")
    print(f"{autonomia_segundos:.0f} segundos")
    
    # Consumo de energia em vôo
    print(f"\n ANÁLISE DE CONSUMO:")
    print(f"Energia inicial: {df['Energia_%'].iloc[0]:.2f}%")
    print(f"Energia final: {df['Energia_%'].iloc[-1]:.2f}%")
    consumo_ciclos = df['Energia_%'].iloc[0] - df['Energia_%'].iloc[-1]
    print(f"Consumo total ({NUM_CICLOS} ciclos): {consumo_ciclos:.2f}%")
    consumo_medio = consumo_ciclos / NUM_CICLOS
    print(f"Consumo médio por ciclo: {consumo_medio:.4f}%")
    
    # Margem de segurança
    margem_seguranca = energia_percentual - ENERGIA_MINIMA
    print(f"\n MARGEM DE SEGURANÇA:")
    print(f"Energia Atual: {energia_percentual:.2f}%")
    print(f"Energia Mínima: {ENERGIA_MINIMA}%")
    print(f"Margem: {margem_seguranca:.2f}%")
    
    if margem_seguranca >= 10:
        print(f"Status: EXCELENTE")
    elif margem_seguranca >= 5:
        print(f"Status: BOA")
    else:
        print(f"Status: CRÍTICA")
    
    # Recomendações
    print(f"\n RECOMENDAÇÕES:")
    if autonomia_minutos < AUTONOMIA_MINIMA:
        print(f"Autonomia ABAIXO do limite. Recarregar antes de decolar.")
    elif autonomia_minutos < AUTONOMIA_MINIMA * 1.2:
        print(f"Autonomia PRÓXIMA ao limite. Considere recarregar.")
    else:
        print(f"Autonomia ADEQUADA para operação.")
    
    if energia_percentual < 85:
        print(f"Energia abaixo de 85%. Recarregar para operação segura.")
    elif energia_percentual < 90:
        print(f"Energia sub-ótima. Considere carregar mais.")
    else:
        print(f"Energia em nível ótimo.")
    
    print("\n" + "="*70 + "\n")
    
    return {
        'energia_kwh': energia_kwh,
        'consumo_total_kw': consumo_total_kw,
        'autonomia_horas': autonomia_horas,
        'autonomia_minutos': autonomia_minutos,
        'margem_seguranca': margem_seguranca
    }

# Executar análise energética
análise = analisar_energia(df_sensores, df_sensores.iloc[-1])


ANÁLISE ENERGÉTICA DETALHADA

 CAPACIDADE DO SISTEMA:
Capacidade Total: 100 kWh
Carga Atual: 86.87 kWh (86.87%)

 CONSUMO NA DECOLAGEM:
Potência Nominal: 5000 W (5.0 kW)

 PERDAS ENERGÉTICAS:
Taxa de Perda: 15.0%
Perda Absoluta: 0.750 kW

 CONSUMO TOTAL:
Consumo + Perdas: 5.750 kW

 AUTONOMIA ESTIMADA:
15.11 horas
906.47 minutos
54388 segundos

 ANÁLISE DE CONSUMO:
Energia inicial: 89.84%
Energia final: 86.87%
Consumo total (30 ciclos): 2.97%
Consumo médio por ciclo: 0.0990%

 MARGEM DE SEGURANÇA:
Energia Atual: 86.87%
Energia Mínima: 80%
Margem: 6.87%
Status: BOA

 RECOMENDAÇÕES:
Autonomia ADEQUADA para operação.
Energia sub-ótima. Considere carregar mais.




## Célula 7: Decisão Final de Decolagem

In [8]:
print("\n\n" + "#"*70)
print("#" + " "*68 + "#")
print("#" + " "*15 + "DECISÃO FINAL DE DECOLAGEM" + " "*15 + "#")
print("#" + " "*68 + "#")
print("#"*70)

print("\n" + "="*70)

if pode_decolar:
    print(" "*66 )
    print(" "*18 + "PRONTO PARA DECOLAR" + " "*31)
    print(" "*66 + "")
    print("="*70)
    print("\n STATUS OPERACIONAL:")
    print(f"Todos os sistemas verificados")
    print(f"Integridade estrutural: OK")
    print(f"Energia: {df_sensores.iloc[-1]['Energia_%']:.2f}% (Mínimo: {ENERGIA_MINIMA}%)")
    print(f"Pressão dos tanques: {df_sensores.iloc[-1]['Pressao_Tanques_PSI']:.2f} PSI")
    print(f"Módulos críticos: 6/6 operacionais")
    print(f"Autonomia: {análise['autonomia_minutos']:.2f} minutos")
    print("\n AUTORIZAÇÃO: ATIVADA")
    print(f"Data/Hora: {datetime.now().strftime('%d/%m/%Y %H:%M:%S')}")
    print(f"Status: LIBERADO PARA DECOLAGEM IMEDIATA")
else:
    print(" "*66)
    print(" "*16 + "DECOLAGEM ABORTADA" + " "*34)
    print(" "*66)
    print("="*70)
    print(f"\n MOTIVOS DO ABORTO ({len(motivos)} falha(s) detectada(s)):")
    for i, motivo in enumerate(motivos, 1):
        print(f"{i}.{motivo}")
    print("\n AÇÕES RECOMENDADAS:")
    print("1. Recarregar energia se nível insuficiente")
    print("2. Realizar manutenção preventiva")
    print("3. Validar integridade estrutural")
    print("4. Verificar pressão dos sistemas")
    print("5. Reexecutar verificação após correções")

print("\n" + "="*70)
print(f"\n RELATÓRIO GERADO: {datetime.now().strftime('%d/%m/%Y às %H:%M:%S')}")
print(f"CICLOS ANALISADOS: {NUM_CICLOS}")
print(f"CAPACIDADE DO SISTEMA: {CAPACIDADE_BATERIA} kWh")
print(f"AUTONOMIA MÍNIMA REQUERIDA: {AUTONOMIA_MINIMA} minutos")
print("\n" + "#"*70)
print("#" + " "*68 + "#")
print("#" + " "*22 + "FIM DA VERIFICAÇÃO" + " "*28 + "#")
print("#" + " "*68 + "#")
print("#"*70 + "\n")



######################################################################
#                                                                    #
#               DECISÃO FINAL DE DECOLAGEM               #
#                                                                    #
######################################################################

                                                                  
                  PRONTO PARA DECOLAR                               
                                                                  

 STATUS OPERACIONAL:
Todos os sistemas verificados
Integridade estrutural: OK
Energia: 86.87% (Mínimo: 80%)
Pressão dos tanques: 541.75 PSI
Módulos críticos: 6/6 operacionais
Autonomia: 906.47 minutos

 AUTORIZAÇÃO: ATIVADA
Data/Hora: 16/09/2026 01:12:54
Status: LIBERADO PARA DECOLAGEM IMEDIATA


 RELATÓRIO GERADO: 16/09/2026 às 01:12:54
CICLOS ANALISADOS: 30
CAPACIDADE DO SISTEMA: 100 kWh
AUTONOMIA MÍNIMA REQUERIDA: 60 minutos

#################

## Célula 8: Visualização de Dados e Gráficos

In [9]:
# Criar visualizações dos dados
print("\n" + "="*70)
print("VISUALIZAÇÃO DOS DADOS COLETADOS")
print("="*70 + "\n")

# Tabela resumida
print("TABELA DE DADOS COMPLETA:\n")
tabela_resumo = df_sensores[[
    'Ciclo', 'Temperatura_Interna_°C', 'Temperatura_Externa_°C',
    'Integridade_Estrutural', 'Energia_%', 'Pressao_Tanques_PSI'
]].copy()

print(tabela_resumo.to_string(index=False))

print("\n" + "="*70)
print("ESTATÍSTICAS DESCRITIVAS:\n")
print(df_sensores[[
    'Temperatura_Interna_°C', 'Temperatura_Externa_°C',
    'Energia_%', 'Pressao_Tanques_PSI'
]].describe().round(2))

print("\n" + "="*70)
print("Notebook executado com sucesso!")
print("="*70 + "\n")


VISUALIZAÇÃO DOS DADOS COLETADOS

TABELA DE DADOS COMPLETA:

 Ciclo  Temperatura_Interna_°C  Temperatura_Externa_°C  Integridade_Estrutural  Energia_%  Pressao_Tanques_PSI
     1                   25.20                   15.38                       1      89.84               526.65
     2                   23.01                  -17.38                       1      89.73               555.37
     3                   25.58                   -7.80                       1      97.07               515.01
     4                   33.25                   -3.00                       1      91.01               524.14
     5                   18.62                   -8.32                       1      91.62               507.44
     6                   12.68                    9.84                       1      91.51               495.65
     7                   52.88                  -21.25                       1      94.98               498.74
     8                   26.10                   -

## Notas Finais

### Resumo da Execução:

Este notebook implementou com sucesso:

1. ✅ **Geração de Dados Simulados**
   - 30 ciclos de leitura de sensores
   - Dados realistas com distribuição apropriada
   - Armazenamento em DataFrame Pandas

2. ✅ **Interpretação de Dados**
   - Análise de temperatura interna e externa
   - Verificação de integridade estrutural
   - Monitoramento de níveis de energia
   - Análise de pressão dos tanques
   - Status dos 6 módulos críticos

3. ✅ **Algoritmo de Verificação**
   - 7 etapas de validação
   - Decisão binária (Decolar/Abortar)
   - Motivos documentados para aborto

4. ✅ **Análise Energética**
   - Cálculo de autonomia
   - Análise de consumo
   - Margem de segurança
   - Recomendações operacionais

---

**Projeto**: Sistema de Verificação de Decolagem - FIAP CCOMP  
**Data**: Setembro 2026  
**Versão**: 1.0